# Correlated-read ramp fitting

JWST detectors are read non-destructively several times ("groups") during each integration.
The slope of the accumulated signal is the count rate. Adjacent group differences are
correlated, because each read's noise enters two differences. `ts.ramp_fitting.fit_ramps`
fits slopes with the full covariance of the differences (the method of
[arXiv:2305.06240](https://arxiv.org/abs/2305.06240)), rejects jumps (e.g. cosmic rays), and
returns rates in DN/s with separate Poisson and read-noise variances.

Here we simulate ramps with known rates and check the fitted rates and their errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from transitspectroscopy import ramp_fitting

## Simulate ramps

1000 integrations of 20 groups on a 4 × 4-pixel patch. Photons arrive with Poisson
statistics, and every read adds independent read noise. A few integrations get a cosmic-ray
jump.

In [ ]:
rng = np.random.default_rng(3)
nints, ngroups, shape = 1000, 20, (4, 4)
group_time, gain, read_noise = 1.0, 2.0, 12.0     # s, e-/DN, e- per read
rate_e = np.full(shape, 200.0)                    # true rate, electrons/s

increments = rng.poisson(rate_e * group_time, (nints, ngroups, *shape))
electrons = np.cumsum(increments, axis=1) + rng.normal(0, read_noise, (nints, ngroups, *shape))
data = electrons / gain                           # DN

jumps = rng.choice(nints, 20, replace=False)
data[jumps, 10:, 1, 2] += 5000.                   # cosmic rays from group 10 onwards
print(data.shape)

## Fit

`gain` is in electrons/DN and `read_noise` in RMS electrons **per read**. With real JWST
data use `ts.jwst.load(..., ramp_fit_method='sparta')` instead; it reads both from the CRDS
reference files (see {doc}`../guides/ramp-fitting`).

In [ ]:
result = ramp_fitting.fit_ramps(data, group_time=group_time, gain=gain,
                                read_noise=read_noise, sigma=5.)
rates, errors = result['data'], result['err']     # DN/s
true_rate = rate_e / gain
print('mean fitted rate:', rates.mean().round(3), 'DN/s; true rate:', true_rate.mean(), 'DN/s')
print('groups rejected in jump pixels:', result['n_rejected'][jumps, 1, 2].mean())

## Are the errors right?

If the error bars are correct, `(rate - truth) / error` should have unit standard deviation.
We compare against a simple estimate from the first and last groups.

In [ ]:
pulls = ((rates - true_rate) / errors).ravel()
last_minus_first = (data[:, -1] - data[:, 0]) / ((ngroups - 1) * group_time)
print('std of pulls:', pulls.std().round(3))
print('scatter, correlated fit   :', rates[:, 0, 0].std().round(4), 'DN/s')
print('scatter, last minus first :', np.delete(last_minus_first[:, 0, 0], jumps).std().round(4), 'DN/s')

plt.hist(pulls, bins=60, density=True, alpha=.6, label='fit')
x = np.linspace(-4, 4, 200)
plt.plot(x, np.exp(-x**2 / 2) / np.sqrt(2 * np.pi), 'k', label='unit Gaussian')
plt.xlabel('(rate - truth) / error'); plt.legend();

The fitted slopes are unbiased, the errors match the scatter, the jumps are rejected,
and the fit is more precise than a last-minus-first estimate.